# ESM-2 3B final-layer residue embeddings

Select an L4 or A100 GPU runtime. The pinned model produces float32 `[L,2560]` matrices, packaged into validated ZIP checkpoints of 500 proteins each for efficient Drive download and resume.


In [ ]:
RUN_MODE = "smoke"  # "smoke" or "full"
DRIVE_ROOT = "/content/drive/MyDrive/dynamic_protein_router/esm2_3b"
MANIFEST_PATH = f"{DRIVE_ROOT}/input/esm2_3b_input_manifest.csv"
OUTPUT_ROOT = f"{DRIVE_ROOT}/embeddings"
CHECKPOINT_SIZE = 500
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be smoke or full")


In [ ]:
%pip install -q "transformers==4.48.3" "accelerate>=1.2,<2" pyarrow
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
if not Path(MANIFEST_PATH).is_file():
    raise FileNotFoundError(f"Upload the prepared manifest to {MANIFEST_PATH}")


In [ ]:
"""Resumable final-layer ESM-2 residue extraction for a Colab GPU."""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import tempfile
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "facebook/esm2_t36_3B_UR50D"
MODEL_REVISION = "7bfbb6ae874b2d2948a5ecb2a62fbad7e9083c32"
OUTPUT_WIDTH = 2560
MAX_SEQUENCE_LENGTH = 1022
EXTRACTION_CONFIG = {
    "representation": "esm2_final_hidden_state",
    "layer": 36,
    "special_tokens_removed": True,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "inference_dtype": "float16",
    "output_dtype": "float32",
}
CHECKPOINT_SIZE = 500
CHECKPOINT_FORMAT_VERSION = 1


def sequence_sha256(sequence: str) -> str:
    return hashlib.sha256(sequence.encode()).hexdigest()


def expected_metadata(row: Any) -> dict[str, object]:
    length = int(row.sequence_length)
    return {
        "protein_id": str(row.protein_id),
        "sequence_sha256": str(row.sequence_sha256),
        "model_sequence_sha256": str(row.sequence_sha256),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "shape": [length, OUTPUT_WIDTH],
        "dtype": "float32",
        "extraction_config": EXTRACTION_CONFIG,
    }


def valid_result(path: Path, row: Any) -> bool:
    try:
        with np.load(path, allow_pickle=False) as archive:
            if set(archive.files) != {"single", "metadata"}:
                return False
            values = archive["single"]
            metadata = json.loads(str(archive["metadata"].item()))
        return (
            values.dtype == np.float32
            and values.shape == (int(row.sequence_length), OUTPUT_WIDTH)
            and np.isfinite(values).all()
            and metadata == expected_metadata(row)
        )
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return False


def atomic_npz(path: Path, values: np.ndarray, metadata: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        np.savez_compressed(
            handle,
            single=values.astype(np.float32, copy=False),
            metadata=np.array(json.dumps(metadata, sort_keys=True)),
        )
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def checkpoint_path(output_root: Path, checkpoint_index: int) -> Path:
    return output_root / "checkpoints" / f"esm2_3b_checkpoint_{checkpoint_index:05d}.zip"


def expected_checkpoint_metadata(checkpoint_index: int, rows: Any) -> dict[str, object]:
    return {
        "format_version": CHECKPOINT_FORMAT_VERSION,
        "representation": "esm2_3b",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "extraction_config": EXTRACTION_CONFIG,
        "checkpoint_index": checkpoint_index,
        "members": [
            {
                "protein_id": str(row.protein_id),
                "sequence_sha256": str(row.sequence_sha256),
                "sequence_length": int(row.sequence_length),
            }
            for row in rows.itertuples(index=False)
        ],
    }


def valid_checkpoint(path: Path, checkpoint_index: int, rows: Any) -> bool:
    expected_metadata = expected_checkpoint_metadata(checkpoint_index, rows)
    expected_members = {f"{member['sequence_sha256']}.npz" for member in expected_metadata["members"]}
    try:
        with zipfile.ZipFile(path) as archive:
            names = set(archive.namelist())
            if names != {"checkpoint.json", *expected_members}:
                return False
            metadata = json.loads(archive.read("checkpoint.json"))
        return metadata == expected_metadata
    except (OSError, ValueError, KeyError, json.JSONDecodeError, zipfile.BadZipFile):
        return False


def atomic_checkpoint(
    path: Path, checkpoint_index: int, rows: Any, staging_root: Path
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata = expected_checkpoint_metadata(checkpoint_index, rows)
    with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.writestr("checkpoint.json", json.dumps(metadata, indent=2, sort_keys=True))
            for member in metadata["members"]:
                archive.write(staging_root / f"{member['sequence_sha256']}.npz", arcname=f"{member['sequence_sha256']}.npz")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


class ESM2FinalLayerExtractor:
    """Pinned ESM-2 3B final hidden-state extractor."""

    def __init__(self, *, device: str = "cuda", model: Any = None, tokenizer: Any = None) -> None:
        import torch

        self.device = torch.device(device)
        if (model is None) != (tokenizer is None):
            raise ValueError("model and tokenizer must be supplied together")
        if model is None:
            from transformers import AutoModel, AutoTokenizer

            tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
            model = AutoModel.from_pretrained(
                MODEL_ID,
                revision=MODEL_REVISION,
                torch_dtype=torch.float16 if self.device.type == "cuda" else torch.float32,
                # ESM-2 3B checkpoints can be left on PyTorch's ``meta`` device
                # by the low-memory loader in current Colab combinations of
                # Transformers/Accelerate.  Load materialized weights first,
                # then move the complete module to CUDA below.
                low_cpu_mem_usage=False,
            )
        self.model = model.to(self.device).eval()
        self.tokenizer = tokenizer

    def extract(self, sequence: str) -> np.ndarray:
        import torch

        sequence = sequence.upper()
        if not sequence or any(character.isspace() for character in sequence):
            raise ValueError("sequence must be non-empty uppercase text without whitespace")
        if len(sequence) > MAX_SEQUENCE_LENGTH:
            raise ValueError(f"sequence exceeds the {MAX_SEQUENCE_LENGTH}-residue limit")
        tokens = self.tokenizer(
            sequence,
            return_tensors="pt",
            return_special_tokens_mask=True,
            add_special_tokens=True,
        )
        special = tokens.pop("special_tokens_mask").bool()
        attention = tokens["attention_mask"].bool()
        keep = attention & ~special
        if int(keep.sum()) != len(sequence):
            raise ValueError("ESM-2 tokenization did not preserve one token per residue")
        tokens = {name: value.to(self.device) for name, value in tokens.items()}
        with torch.inference_mode():
            hidden = self.model(**tokens).last_hidden_state
        values = hidden[keep.to(hidden.device)].detach().float().cpu().numpy()
        if values.shape != (len(sequence), OUTPUT_WIDTH) or not np.isfinite(values).all():
            raise ValueError(f"invalid ESM-2 embedding shape or values: {values.shape}")
        return values.astype(np.float32, copy=False)


def load_manifest(path: Path):
    import pandas as pd

    frame = pd.read_csv(path)
    required = {"protein_id", "sequence", "sequence_sha256", "sequence_length", "eligible"}
    if missing := required - set(frame):
        raise ValueError(f"manifest is missing columns: {sorted(missing)}")
    frame = frame.loc[frame.eligible.map(lambda value: str(value).lower() == "true")].copy()
    if frame.empty or frame.protein_id.duplicated().any() or frame.sequence_sha256.duplicated().any():
        raise ValueError("eligible manifest identities must be non-empty and unique")
    if not frame.sequence.map(sequence_sha256).equals(frame.sequence_sha256):
        raise ValueError("manifest sequence hashes do not match")
    if not frame.sequence.str.len().equals(frame.sequence_length.astype(int)):
        raise ValueError("manifest sequence lengths do not match")
    if (frame.sequence_length.astype(int) > MAX_SEQUENCE_LENGTH).any():
        raise ValueError("eligible manifest includes an over-length sequence")
    return frame.sort_values(["sequence_length", "sequence_sha256"]).reset_index(drop=True)


def run(
    manifest: Path, output_root: Path, *, max_proteins: int = 0, checkpoint_size: int = CHECKPOINT_SIZE
) -> dict[str, object]:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("the production ESM-2 worker requires a CUDA Colab runtime")
    rows = load_manifest(manifest)
    if max_proteins:
        rows = rows.head(max_proteins)
    if checkpoint_size < 1:
        raise ValueError("checkpoint_size must be positive")
    output_root.mkdir(parents=True, exist_ok=True)
    extractor = ESM2FinalLayerExtractor(device="cuda")
    completed = 0
    checkpoint_total = (len(rows) + checkpoint_size - 1) // checkpoint_size
    for checkpoint_index, start in enumerate(range(0, len(rows), checkpoint_size)):
        batch = rows.iloc[start : start + checkpoint_size].copy()
        destination = checkpoint_path(output_root, checkpoint_index)
        if not valid_checkpoint(destination, checkpoint_index, batch):
            with tempfile.TemporaryDirectory(prefix="esm2_3b_checkpoint_") as temporary_directory:
                staging_root = Path(temporary_directory)
                for row in batch.itertuples(index=False):
                    values = extractor.extract(str(row.sequence))
                    member = staging_root / f"{row.sequence_sha256}.npz"
                    atomic_npz(member, values, expected_metadata(row))
                    if not valid_result(member, row):
                        raise ValueError(f"staged ESM-2 result failed validation: {row.protein_id}")
                atomic_checkpoint(destination, checkpoint_index, batch, staging_root)
        if not valid_checkpoint(destination, checkpoint_index, batch):
            raise ValueError(f"written ESM-2 checkpoint failed validation: {destination.name}")
        completed += len(batch)
        progress = {
            "status": "running",
            "completed": completed,
            "expected": len(rows),
            "checkpoint_completed": checkpoint_index + 1,
            "checkpoint_total": checkpoint_total,
            "checkpoint_file": destination.name,
            "protein_id": str(batch.iloc[-1].protein_id),
            "updated_at_utc": datetime.now(UTC).isoformat(),
        }
        (output_root / "progress.json").write_text(json.dumps(progress, indent=2) + "\n")
        print(json.dumps(progress), flush=True)
    progress.update(status="complete")
    (output_root / "progress.json").write_text(json.dumps(progress, indent=2) + "\n")
    return progress


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--output-root", type=Path, required=True)
    parser.add_argument("--max-proteins", type=int, default=0)
    parser.add_argument("--checkpoint-size", type=int, default=CHECKPOINT_SIZE)
    args = parser.parse_args()
    print(
        json.dumps(
            run(
                args.manifest,
                args.output_root,
                max_proteins=args.max_proteins,
                checkpoint_size=args.checkpoint_size,
            ),
            indent=2,
        )
    )


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required")
print(torch.cuda.get_device_name(0))
run(
    Path(MANIFEST_PATH),
    Path(OUTPUT_ROOT),
    max_proteins=3 if RUN_MODE == "smoke" else 0,
    checkpoint_size=CHECKPOINT_SIZE,
)
